In [1]:
!pip uninstall -y edgar edgartools
!pip list | grep edgar
!pip install edgartools --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 216.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 355.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 193.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from edgar import set_identity, Company
from pathlib import Path
import time
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_dir = Path("/content/drive/MyDrive/data/Sections")
data_dir.mkdir(parents=True, exist_ok=True)

set_identity("Rens Gerritsen 658549rg@eur.nl")

# tickers = ["0000001750", "0000002969", "0000002488"]

print(os.listdir("/content/drive/MyDrive/data"))

ciks = pd.read_csv("/content/drive/MyDrive/data/sp500_historical_ciks.csv")

print(f"Loaded {len(ciks)} CIKs")
start_yr = 2020

MAX_WORKERS = 5   # ⚠️ safe range: 3–8 (avoid SEC throttling)

def get_text_from_section(section):
    if section is None:
        return None
    if isinstance(section, str):
        return section
    if hasattr(section, 'text'):
        return section.text
    return str(section)


def extract_section_by_regex(text, start_pattern, end_pattern):
    match = re.search(start_pattern, text, re.IGNORECASE | re.DOTALL)
    if not match:
        return None
    start = match.end()
    end_match = re.search(end_pattern, text[start:], re.IGNORECASE | re.DOTALL)
    end = start + end_match.start() if end_match else start + 50000
    return text[start:end].strip()


def parse_sections_list(sections):
    ITEM1_PATTERNS  = [r'^item\s*1$', r'^item\s*1[\.\s]+business']
    ITEM1A_PATTERNS = [r'^item\s*1a', r'^item\s*1a[\.\s]+risk']
    ITEM7_PATTERNS  = [r'^item\s*7$', r'^item\s*7[\.\s]+management']

    strategy_text = None
    risk_text = None
    mgmt_text = None

    for s in sections:
        if isinstance(s, dict):
            label = s.get('item', s.get('title', s.get('name', '')))
            text  = s.get('text', s.get('content', ''))
        elif hasattr(s, 'title'):
            label = s.title
            text  = get_text_from_section(s)
        else:
            continue

        label_clean = label.lower().strip()

        if not strategy_text and any(re.match(p, label_clean) for p in ITEM1_PATTERNS):
            strategy_text = text
        elif not risk_text and any(re.match(p, label_clean) for p in ITEM1A_PATTERNS):
            risk_text = text
        elif not mgmt_text and any(re.match(p, label_clean) for p in ITEM7_PATTERNS):
            mgmt_text = text

    return strategy_text, risk_text, mgmt_text


def process_ticker(ticker):
    try:
        company = Company(ticker)

        company_name = re.sub(r'[\\/*?:"<>|]', "", company.name)
        company_dir = data_dir / company_name
        company_dir.mkdir(parents=True, exist_ok=True)

        filings = company.get_filings(form="10-K")
        for filing in filings:
            year = filing.filing_date.year
            if year < start_yr:
                continue

            strat_path = company_dir / f"{year}_item1_business.txt"
            risk_path  = company_dir / f"{year}_item1a_risk.txt"
            mgmt_path  = company_dir / f"{year}_item7_mda.txt"

            if strat_path.exists() and risk_path.exists() and mgmt_path.exists():
                continue

            try:
                tenk = filing.obj()

                strategy_text = get_text_from_section(tenk.business)
                risk_text     = get_text_from_section(tenk.risk_factors)
                mgmt_text     = get_text_from_section(tenk.management_discussion)

            except:
                strategy_text = risk_text = mgmt_text = None

            # Fallback: sections()
            if not any([strategy_text, risk_text, mgmt_text]):
                try:
                    sections = filing.sections()
                    strategy_text, risk_text, mgmt_text = parse_sections_list(sections)
                except:
                    pass

            # Fallback: raw text (expensive)
            if not any([strategy_text, risk_text, mgmt_text]):
                try:
                    raw = filing.text()
                    strategy_text = extract_section_by_regex(
                        raw, r'item\s*1[\.\s]+business', r'item\s*1a[\.\s]'
                    )
                    risk_text = extract_section_by_regex(
                        raw, r'item\s*1a[\.\s]+risk', r'item\s*1b[\.\s]'
                    )
                    mgmt_text = extract_section_by_regex(
                        raw, r'item\s*7[\.\s]+management', r'item\s*7a[\.\s]'
                    )
                except:
                    pass

            # Save
            if strategy_text:
                strat_path.write_text(strategy_text, encoding="utf-8")
            if risk_text:
                risk_path.write_text(risk_text, encoding="utf-8")
            if mgmt_text:
                mgmt_path.write_text(mgmt_text, encoding="utf-8")

            time.sleep(0.2)  # ⚠️ SEC-safe delay

        return f"✅ Done: {company_name}"

    except Exception as e:
        return f"❌ Error {ticker}: {e}"


def download_filings_threaded():
    cik_list = ciks['cik'].astype(str).str.zfill(10).tolist()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_ticker, t) for t in cik_list]

        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing companies"):
            result = future.result()
            # optional: print errors only
            if "❌" in result:
                print(result)


download_filings_threaded()

['Sections', 'sp500_historical_ciks.csv']
Loaded 1046 CIKs


Processing companies:   1%|          | 8/1046 [00:03<08:24,  2.06it/s]WARNING:edgar.core:TenK falling back to legacy parser for 'Item 1' (filing: 0000002488-26-000021). New parser sections available: ['mda']. This fallback will be removed in v6.0.


Processing companies:   1%|          | 10/1046 [04:26<21:27:30, 74.57s/it]WARNING:edgar.core:TenK falling back to legacy parser for 'Item 1' (filing: 0000004904-25-000027). New parser sections available: ['part_ii_item_5', 'part_ii_item_6', 'part_ii_item_7', 'part_ii_item_7a', 'part_ii_item_8', 'part_iv_item_7', 'part_iv_item_8', 'part_ii_item_9', 'part_ii_item_9a', 'part_ii_item_9b', 'part_ii_item_9c', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_i_item_1a', 'part_i_item_1b', 'part_i_item_1c', 'part_i_item_2', 'part_i_item_3', 'part_i_item_4']. This fallback will be removed in v6.0.


Processing companies:   1%|          | 10/1046 [10:08<17:31:15, 60.88s/it]
